# NOTE-RL v2 - Advanced Resource-Constrained Medical QA with Diagnostic Self-Distillation
**Single notebook, Kaggle 2xT4, hardened for the 12-hour session limit.**

This version fixes the dataset error and adds research-grade machinery on top of the original pipeline:

**Core pipeline:** SFT logit-classifier -> difficulty-aware mining -> 3-gate pseudo-labeling (self-consistency + verifier + confidence) -> ChromaDB note memory -> soft-label KD -> verifier-gated synthesis -> GRPO-KD -> born-again.

**New in v2:**
- **Robust loader** - tries a prioritized list of *verified* Hub IDs; carves a val split if one is missing; schema-agnostic normalizer.
- **Multi-level de-duplication** - embedding near-dup filtering for (a) memory notes, (b) synthetic questions, (c) the pseudo pool, plus **test-set decontamination** (drop any train/synthetic item too similar to a test question - prevents leakage).
- **Calibration** - post-hoc **temperature scaling** fit on val; reports **ECE before/after** and saves both reliability diagrams.
- **Statistical rigor** - **bootstrap 95% CIs** for test accuracy and macro-F1.
- **12h budget guard** - a global deadline; every phase checks remaining time and stops gracefully, still exporting all artifacts.
- **Confidence-weighted KD** - per-example KD weight from teacher confidence.
- **Configurable teacher** - default ungated `Qwen2.5-3B-Instruct` (no HF token needed); flip to `7B` 4-bit for a stronger teacher.
- **Optional ablation harness** - re-runs the pipeline with components toggled to fill the ablation table automatically (time-guarded, off by default).

> **Secrets:** optional. Add `HF_TOKEN` / `WANDB_API_KEY` via *Add-ons -> Secrets* only if you use a gated model. The default teacher is ungated.
> **Budget:** `cfg.fast=True` for a quick smoke test; `False` for the full run. `cfg.time_budget_hours` caps wall-clock.


In [ ]:
# === Secrets (optional, NO hardcoded keys) ===
import os
def _load_secret(name):
    v = os.environ.get(name)
    if v: return v
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v: os.environ[name] = v; return v
    except Exception:
        pass
    return None
HF_TOKEN = _load_secret("HF_TOKEN")
WANDB_API_KEY = _load_secret("WANDB_API_KEY")
os.environ["WANDB_MODE"] = "online" if WANDB_API_KEY else "offline"
print("HF_TOKEN:", bool(HF_TOKEN), "(optional) | WANDB:", bool(WANDB_API_KEY), "(optional)")


In [ ]:
# === Installs ===
import sys, subprocess
def pip(*p): subprocess.check_call([sys.executable,"-m","pip","install","-q",*p])
def ensure(mod, pkg=None):
    try: __import__(mod)
    except Exception: pip(pkg or mod)
for mod, pkg in [("datasets","datasets"),("transformers","transformers>=4.41"),
                 ("accelerate","accelerate"),("sklearn","scikit-learn"),
                 ("matplotlib","matplotlib"),("bitsandbytes","bitsandbytes"),
                 ("sentence_transformers","sentence-transformers"),
                 ("pynvml","pynvml"),("pandas","pandas"),
                 ("peft","peft"),("huggingface_hub","huggingface_hub")]:
    ensure(mod,pkg)
print("installs ok (chromadb intentionally not used: numpy vector store instead)")


In [ ]:
# === Imports & devices ===
import os, gc, json, math, random, re, time, threading, warnings
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset as TorchDataset
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
assert torch.cuda.is_available(), "CUDA required."
N_GPU = torch.cuda.device_count()
STUDENT_DEVICE = torch.device("cuda:0")
TEACHER_DEVICE = torch.device("cuda:1" if N_GPU>1 else "cuda:0")
print("GPUs="+str(N_GPU)+" | student="+torch.cuda.get_device_name(0)+
      " | teacher="+(torch.cuda.get_device_name(1) if N_GPU>1 else torch.cuda.get_device_name(0)))
OPTIONS = list("ABCD")
def free_gpu():
    gc.collect(); torch.cuda.empty_cache()
    try: torch.cuda.ipc_collect()
    except Exception: pass


In [ ]:
# === Configuration (OOM-safe: Qwen 3B student+teacher, QLoRA, tiny batches) ===
@dataclass
class CFG:
    seed: int = 42
    fast: bool = False   # FULL run (7B smoke test passed: no OOM)   # FULL run: 4000 labels + 500/1000/2000 efficiency sweep
    time_budget_hours: float = 11.0
    time_reserve_hours: float = 0.5

    dataset_candidates: tuple = (
        "sabarna07/MedQA-USMLE-4-options-hf",
        "GBaker/MedQA-USMLE-4-options-hf",
        "openlifescienceai/MedQA-USMLE-4-options-hf",
    )
    local_data_dir: str = ""
    seed_labeled_n: int = 2000
    unlabeled_n: int = 4000
    val_n: int = 1000
    test_n: int = 1273

    # student = Qwen2.5-3B-Instruct, 4-bit QLoRA
    student_model: str = "Qwen/Qwen2.5-7B-Instruct"
    use_qlora: bool = True
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_targets: tuple = ("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
    max_src_len: int = 448
    grad_clip: float = 1.0
    label_smoothing: float = 0.0
    option_shuffle_augment: bool = True
    n_aug_copies: int = 1
    sft_epochs: int = 1
    sft_bs: int = 4
    sft_grad_accum: int = 8
    sft_lr: float = 1e-4
    sft_warmup_ratio: float = 0.06
    sft_weight_decay: float = 0.01

    # teacher = SAME-FAMILY 3B (self-distillation), 4-bit, small batches -> OOM-safe
    # (To try 7B distillation if you have headroom: set teacher_model below and keep
    #  teacher_gen_mb=2, teacher_score_bs=2 -- but 3B is the safe default.)
    teacher_model: str = "FreedomIntelligence/HuatuoGPT-o1-7B"  # ungated, Qwen2.5-7B medical reasoning
    teacher_max_new_tokens: int = 8
    teacher_note_tokens: int = 80
    teacher_temperature: float = 0.7
    teacher_cot: bool = True          # let the reasoning teacher think before answering (vote gate)
    teacher_cot_tokens: int = 512  # o1-style teacher needs room to finish reasoning     # max reasoning tokens per sample
    teacher_max_len: int = 640      # max prompt tokens fed to the teacher
    teacher_score_bs: int = 8       # forward (option-prob) batch
    teacher_gen_mb: int = 2         # prompts per generation batch (x sc_samples sequences)

    mine_alpha: float = 0.5
    mine_margin_w: float = 0.3
    hard_per_iter: int = 150

    num_iters: int = 1
    sc_samples: int = 5
    sc_vote_frac: float = 0.6
    vote_smooth: float = 0.5   # add-alpha smoothing for vote-distribution soft targets
    conf_gate: float = 0.55
    use_verifier_gate: bool = False  # replaced by CoT-vote agreement gate
    pseudo_max_per_iter: int = 2500

    kd_temperature: float = 3.0
    kd_alpha: float = 0.5
    kd_conf_weight: bool = True
    kd_epochs: int = 2
    kd_lr: float = 5e-5
    kd_bs: int = 4

    embed_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    dedup_note_cos: float = 0.92
    dedup_synth_cos: float = 0.95
    decontaminate: bool = True
    decon_cos: float = 0.95
    retrieve_k: int = 2
    max_notes: int = 200   # cap note generations (speed)

    synth_per_iter: int = 80
    rewrites_per_q: int = 2
    synth_max_new_tokens: int = 128

    grpo_enable: bool = True
    grpo_group: int = 6
    grpo_prompts: int = 256
    grpo_lr: float = 5e-6
    grpo_kl: float = 0.1
    grpo_epochs: int = 1

    ba_enable: bool = False
    ba_epochs: int = 1
    ba_lr: float = 2e-5
    ba_alpha: float = 0.3

    calibrate_temperature: bool = True
    bootstrap_n: int = 2000

    eval_bs: int = 8
    ece_bins: int = 10
    out_dir: str = "./paper_artifacts_v3_qwen7b"
    run_ablations: bool = False
    run_efficiency_sweep: bool = True
    run_reasoning_eval: bool = True       # UNIQUE: note-conditioned self-consistency + agreement-calibrated abstention
    reason_mem_n: int = 256               # gold items used to build the rationale memory
    reason_test_n: int = 500              # test subset for the SC reasoning eval
    reason_samples: int = 5               # CoT chains per question (self-consistency)
    reason_cot_tokens: int = 256
    reason_retrieve_k: int = 3
    sweep_sizes: tuple = (500, 1000)   # extra SFT-only points for the label-efficiency curve

    push_to_hub: bool = False
    hub_model_id: str = ""
    hub_private: bool = False
    fp16: bool = True

    def apply_fast(self):
        if self.fast:
            self.seed_labeled_n=500; self.unlabeled_n=400; self.val_n=300; self.test_n=300
            self.sft_epochs=1; self.num_iters=1; self.hard_per_iter=60
            self.pseudo_max_per_iter=120; self.synth_per_iter=24; self.kd_epochs=1
            self.grpo_prompts=32; self.grpo_group=4; self.bootstrap_n=500
            self.teacher_cot_tokens=384; self.sweep_sizes=(150, 300); self.reason_mem_n=32; self.reason_test_n=40; self.reason_samples=3; self.reason_cot_tokens=128
            self.time_budget_hours=3.0
        return self

cfg = CFG().apply_fast()
os.makedirs(cfg.out_dir, exist_ok=True)
random.seed(cfg.seed); np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed); torch.cuda.manual_seed_all(cfg.seed)
print("fast:", cfg.fast, "| student:", cfg.student_model, "| teacher:", cfg.teacher_model)


In [ ]:
# === Wall-clock budget guard ===
START_TIME = time.time()
DEADLINE = START_TIME + cfg.time_budget_hours*3600
RESERVE = cfg.time_reserve_hours*3600
def time_left(): return DEADLINE - time.time()
def have_time(need_sec=0.0): return time_left() > (RESERVE + need_sec)
def elapsed_h(): return (time.time()-START_TIME)/3600
print("deadline set; hours available:", round((DEADLINE-time.time())/3600,2))


In [ ]:
# === Optional W&B ===
USE_WANDB = bool(os.environ.get("WANDB_API_KEY")); run=None
if USE_WANDB:
    try:
        import wandb
        run = wandb.init(project="note-rl-medqa-v2", name="note_rl_v2",
                         config=asdict(cfg), reinit=True)
        print("W&B:", getattr(run,"url",None))
    except Exception as e:
        USE_WANDB=False; print("W&B off:", e)
def wlog(d):
    if USE_WANDB and run is not None:
        try: run.log(d)
        except Exception: pass


## 1) Robust data loading & schema-agnostic normalization
Tries verified Hub IDs in order; the first that loads wins. Handles `sent1/ending*/label`, list/dict options, `answer_idx`, and full-text answers. If a `validation` split is missing, one is carved from the train tail.

In [ ]:
# === Load MedQA (BULLETPROOF: direct HTTP -> hf_hub -> local -> resolver) ===
# Why: on some Kaggle images load_dataset(repo) raises DatasetNotFoundError even with
# Internet ON (a datasets/huggingface_hub version mismatch). Method A below downloads the
# raw data files over plain HTTP and never touches those resolvers, so it can't hit that bug.
from datasets import Dataset, load_dataset
import urllib.request, json, glob
HF_TOKEN = globals().get("HF_TOKEN", None)

# ---------- normalizer ----------
def _get_options(ex):
    if all(("ending"+str(i)) in ex for i in range(4)):
        return [str(ex["ending"+str(i)]).strip() for i in range(4)]
    o = ex.get("options")
    if isinstance(o, dict):
        return [str(o.get(k, o.get(k.lower(),""))).strip() for k in OPTIONS]
    if isinstance(o, list) and len(o) >= 4:
        if isinstance(o[0], dict) and "value" in o[0]:
            o = [d["value"] for d in sorted(o, key=lambda x: x.get("key",""))]
        return [str(x).strip() for x in o[:4]]
    for ks in [("opa","opb","opc","opd")]:
        if all(k in ex for k in ks): return [str(ex[k]).strip() for k in ks]
    return None
def normalize_example(ex):
    q = ex.get("sent1") or ex.get("question") or ex.get("stem") or ex.get("prompt") or ""
    if ex.get("sent2") and str(ex["sent2"]).strip() not in ("", "?"):
        q = (str(q).strip()+"\n"+str(ex["sent2"]).strip()).strip()
    opts = _get_options(ex) or ["","","",""]
    lab = ex.get("answer_idx", ex.get("label", ex.get("answer", None)))
    ans = None
    if isinstance(lab, int): ans = OPTIONS[lab] if 0<=lab<4 else None
    elif lab is not None and str(lab).strip().isdigit(): ans = OPTIONS[int(str(lab).strip())]
    elif isinstance(lab, str) and lab.strip().upper() in OPTIONS: ans = lab.strip().upper()
    elif isinstance(lab, str):
        for i,o in enumerate(opts):
            if o and o.strip().lower() == lab.strip().lower(): ans = OPTIONS[i]; break
    if ans is None: ans = "A"
    return {"question": str(q).strip(), "options": opts, "answer": ans}
def normalize_split(split, max_n=None):
    n = len(split) if max_n is None else min(len(split), max_n)
    return Dataset.from_list([normalize_example(split[i]) for i in range(n)])

# ---------- helpers ----------
def _read_records(text):
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, list): return obj
        if isinstance(obj, dict):
            for v in obj.values():
                if isinstance(v, list): return v
            return [obj]
    except Exception:
        pass
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line: continue
        try: rows.append(json.loads(line))
        except Exception: pass
    return rows
def _http_get(url, token=None, timeout=180):
    h = {"User-Agent": "Mozilla/5.0"}
    if token: h["Authorization"] = "Bearer " + token
    with urllib.request.urlopen(urllib.request.Request(url, headers=h), timeout=timeout) as r:
        return r.read().decode("utf-8")
_EXTS = ("parquet","jsonl","json","csv")
def _fmt(p): return {"jsonl":"json","json":"json","parquet":"parquet","csv":"csv"}[p.rsplit(".",1)[-1].lower()]
def _load_file(p): return load_dataset(_fmt(p), data_files=p, split="train")
def _pick(files, keys, avoid=()):
    c = [f for f in files if "." in os.path.basename(f)
         and os.path.basename(f).rsplit(".",1)[-1].lower() in _EXTS
         and any(k in os.path.basename(f).lower() for k in keys)
         and not any(a in os.path.basename(f).lower() for a in avoid)]
    return sorted(c, key=len)[0] if c else None

# Verified direct-file repos (filename per split). JSON files are JSONL: sent1/ending*/label.
HF_DIRECT = [
    ("sabarna07/MedQA-USMLE-4-options-hf", {"train":"train.json","validation":"dev.json","test":"test.json"}),
    ("GBaker/MedQA-USMLE-4-options-hf", {"train":"train.json","validation":"dev.json","test":"test.json"}),
]

raw = None; used_id = None

# ---------- A) plain HTTP direct (most robust) ----------
for repo, fnames in HF_DIRECT:
    try:
        base = "https://huggingface.co/datasets/%s/resolve/main/" % repo
        d = {}
        for split, fn in fnames.items():
            recs = _read_records(_http_get(base + fn, token=HF_TOKEN))
            if not recs: raise ValueError("empty " + fn)
            d[split] = Dataset.from_list(recs)
        raw = d; used_id = repo + " (direct http)"
        print("Loaded via direct HTTP:", repo, "->", {k: len(v) for k, v in d.items()}); break
    except Exception as e:
        print("[http skip]", repo, "->", type(e).__name__, str(e)[:90])

# ---------- B) local files under /kaggle/input (works Internet OFF) ----------
if raw is None:
    search = []
    if getattr(cfg, "local_data_dir", None): search.append(cfg.local_data_dir)
    search += sorted(glob.glob("/kaggle/input/*"))
    allf = []
    for dd in search:
        if os.path.isdir(dd): allf += glob.glob(os.path.join(dd, "**", "*"), recursive=True)
    tr = _pick(allf, ["train"]); va = _pick(allf, ["valid","dev","val"], avoid=["train","test"])
    te = _pick(allf, ["test"], avoid=["train"])
    if tr is not None:
        raw = {"train": _load_file(tr)}
        if va: raw["validation"] = _load_file(va)
        if te: raw["test"] = _load_file(te)
        used_id = "local:" + os.path.dirname(tr)
        print("Loaded LOCAL files from", used_id, "->", {k: len(v) for k, v in raw.items()})

# ---------- C) huggingface_hub direct download ----------
if raw is None:
    try:
        from huggingface_hub import hf_hub_download, list_repo_files
        for cand in cfg.dataset_candidates:
            try:
                files = list_repo_files(cand, repo_type="dataset", token=HF_TOKEN)
                tr = _pick(files, ["train"]); va = _pick(files, ["dev","valid","val"], avoid=["train","test"])
                te = _pick(files, ["test"], avoid=["train"])
                if tr is None: continue
                dl = lambda fn: _load_file(hf_hub_download(cand, fn, repo_type="dataset", token=HF_TOKEN))
                raw = {"train": dl(tr)}
                if va: raw["validation"] = dl(va)
                if te: raw["test"] = dl(te)
                used_id = cand + " (hf_hub)"
                print("Loaded via hf_hub_download:", cand, "->", {k: len(v) for k, v in raw.items()}); break
            except Exception as e:
                print("[hf_hub skip]", cand, "->", type(e).__name__)
    except Exception as e:
        print("huggingface_hub unavailable:", e)

# ---------- D) plain load_dataset resolver (last resort) ----------
if raw is None:
    for cand in cfg.dataset_candidates:
        try:
            raw = load_dataset(cand, token=HF_TOKEN); used_id = cand
            print("Loaded via load_dataset:", cand, "| splits:", list(raw.keys())); break
        except Exception as e:
            print("[resolver skip]", cand, "->", type(e).__name__)

if raw is None:
    raise RuntimeError(
        "\n\nAll load methods failed. Likely no network egress to huggingface.co.\n"
        "  - Offline: 'Add Input' -> add a 'MedQA USMLE 4 options' dataset (loader auto-discovers /kaggle/input),\n"
        "    or set cfg.local_data_dir to its folder and re-run.\n")

# ---------- split handling + build ----------
sp = dict(raw)
assert "train" in sp, "no train split found"
if "validation" not in sp:
    tt = sp["train"].train_test_split(test_size=0.1, seed=cfg.seed)
    sp["train"], sp["validation"] = tt["train"], tt["test"]
if "test" not in sp:
    sp["test"] = sp["validation"]
splits = sp

full_train = normalize_split(splits["train"])
val_ds  = normalize_split(splits["validation"], max_n=cfg.val_n)
test_ds = normalize_split(splits["test"], max_n=cfg.test_n)
idx = list(range(len(full_train))); random.Random(cfg.seed).shuffle(idx)
seed_idx = idx[:cfg.seed_labeled_n]
pool_idx = idx[cfg.seed_labeled_n: cfg.seed_labeled_n+cfg.unlabeled_n]
seed_ds = Dataset.from_list([full_train[i] for i in seed_idx])
unlabeled_ds = Dataset.from_list([full_train[i] for i in pool_idx])
print("dataset:", used_id)
print("seed=%d unlabeled=%d val=%d test=%d" % (len(seed_ds),len(unlabeled_ds),len(val_ds),len(test_ds)))
print("seed label dist:", dict(sorted(Counter(seed_ds["answer"]).items())))
print("sanity - example:", {k:(v[:80] if isinstance(v,str) else v) for k,v in seed_ds[0].items()})


## 2) Prompt, clean letter IDs, option shuffle

In [ ]:
# === Prompt (plain) + student chat wrapper + letter ids ===
from transformers import AutoTokenizer

def format_prompt(q, opts):
    # PLAIN text. Teacher wraps with t_chat(); student wraps with stu_chat().
    return ("You are taking a USMLE-style multiple-choice exam.\n"
            "Choose the single best answer.\n\nQuestion: " + q + "\n\nOptions:\n"
            "A. " + opts[0] + "\nB. " + opts[1] + "\nC. " + opts[2] + "\nD. " + opts[3] + "\n\n"
            "Reply with ONLY one letter: A, B, C, or D.")

def shuffle_options(q, opts, ans, rng):
    perm = list(range(4)); rng.shuffle(perm)
    return [opts[p] for p in perm], OPTIONS[perm.index(OPTIONS.index(ans))]

student_tok = AutoTokenizer.from_pretrained(cfg.student_model, use_fast=True, token=HF_TOKEN)
student_tok.padding_side = "left"        # last token == real last token for every row
student_tok.truncation_side = "left"     # never truncate the answer-eliciting suffix
if student_tok.pad_token is None:
    student_tok.pad_token = student_tok.eos_token

def stu_chat(user, system="You are a medical expert. Answer with a single letter."):
    if hasattr(student_tok, "apply_chat_template"):
        return student_tok.apply_chat_template(
            [{"role":"system","content":system},{"role":"user","content":user}],
            tokenize=False, add_generation_prompt=True)
    return system + "\n\n" + user

def _build_letter_ids(tok):
    out = {}
    for L in OPTIONS:
        c = []
        for v in [L, " " + L]:
            e = tok.encode(v, add_special_tokens=False)
            if len(e) == 1: c.append(e[0])
        out[L] = c or [tok.encode(L, add_special_tokens=False)[0]]
    return out
STU_LETTER_IDS = _build_letter_ids(student_tok)
print("student letter ids:", STU_LETTER_IDS)


In [ ]:
# === Student model: Qwen 4-bit QLoRA + last-token scoring (logits_to_keep=1) ===
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

_sq = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
_base = AutoModelForCausalLM.from_pretrained(
    cfg.student_model, quantization_config=_sq, device_map={"": str(STUDENT_DEVICE)},
    low_cpu_mem_usage=True, token=HF_TOKEN)
_base.config.use_cache = False
_base = prepare_model_for_kbit_training(_base, use_gradient_checkpointing=True)
_lcfg = LoraConfig(r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
                   bias="none", task_type="CAUSAL_LM", target_modules=list(cfg.lora_targets))
student = get_peft_model(_base, _lcfg)
student.print_trainable_parameters()
print("student:", cfg.student_model, "| QLoRA 4-bit")

def _fwd_last(model, input_ids, attention_mask):
    """Forward computing logits for ONLY the final position (avoids full-vocab x seq blowup)."""
    for kw in ("logits_to_keep", "num_logits_to_keep"):
        try:
            return model(input_ids=input_ids, attention_mask=attention_mask, **{kw: 1}).logits[:, -1, :]
        except TypeError:
            continue
    return model(input_ids=input_ids, attention_mask=attention_mask).logits[:, -1, :]

def _letter_logits(last):                  # [B,V] -> [B,4]
    cols = []
    for L in OPTIONS:
        cols.append(torch.logsumexp(torch.stack([last[:, t] for t in STU_LETTER_IDS[L]], 0), 0))
    return torch.stack(cols, 1)

@torch.no_grad()
def student_option_logits(model, prompts, bs=None):
    bs = bs or cfg.eval_bs; model.eval(); out = []
    for i in range(0, len(prompts), bs):
        chats = [stu_chat(p) for p in prompts[i:i+bs]]
        enc = student_tok(chats, return_tensors="pt", padding=True, truncation=True,
                          max_length=cfg.max_src_len).to(STUDENT_DEVICE)
        out.append(_letter_logits(_fwd_last(model, enc.input_ids, enc.attention_mask)).float().cpu())
        del enc
    torch.cuda.empty_cache()
    return torch.cat(out, 0)

def student_probs(model, prompts, bs=None):
    return F.softmax(student_option_logits(model, prompts, bs), -1).numpy()

from peft import get_peft_model_state_dict as _gpsd
INIT_ADAPTER = {k: v.detach().cpu().clone() for k, v in _gpsd(student).items()}
print('captured init adapter:', len(INIT_ADAPTER), 'tensors')


## 3) Metrics, calibration (temperature scaling) & bootstrap CIs

In [ ]:
# === Metrics + temperature scaling + bootstrap ===
def expected_calibration_error(correct, conf, n_bins=10):
    correct=np.asarray(correct,float); conf=np.asarray(conf,float)
    bins=np.linspace(0,1,n_bins+1); e=0.0; n=len(correct)
    for i in range(n_bins):
        lo,hi=bins[i],bins[i+1]
        m=(conf>lo)&(conf<=hi) if i>0 else (conf>=lo)&(conf<=hi)
        if m.sum()==0: continue
        e+=(m.sum()/n)*abs(correct[m].mean()-conf[m].mean())
    return float(e)

def fit_temperature(logits, labels, max_iter=200):
    "Fit scalar T>0 minimizing NLL on cached val logits. logits:[N,4] tensor, labels:[N]."
    logits=torch.tensor(np.asarray(logits),dtype=torch.float32)
    labels=torch.tensor(np.asarray(labels),dtype=torch.long)
    logT=torch.zeros(1, requires_grad=True)
    opt=torch.optim.LBFGS([logT], lr=0.1, max_iter=max_iter)
    def closure():
        opt.zero_grad()
        loss=F.cross_entropy(logits/logT.exp(), labels); loss.backward(); return loss
    opt.step(closure)
    return float(logT.exp().item())

def metrics_from_logits(logits, golds_idx, T=1.0):
    probs=F.softmax(torch.tensor(np.asarray(logits))/T,-1).numpy()
    golds=np.asarray(golds_idx)
    preds=probs.argmax(1); conf=probs.max(1)
    correct=(preds==golds).astype(int)
    acc=accuracy_score(golds,preds)
    f1=f1_score(golds,preds,labels=list(range(4)),average="macro",zero_division=0)
    ece=expected_calibration_error(correct,conf,cfg.ece_bins)
    onehot=np.eye(4)[golds]
    brier=float(np.mean(np.sum((probs-onehot)**2,axis=1)))
    nll=float(-np.mean(np.log(np.clip(probs[np.arange(len(golds)),golds],1e-12,1.0))))
    return dict(acc=acc,f1=f1,ece=ece,brier=brier,nll=nll), preds, conf, correct

def bootstrap_ci(golds_idx, preds, n=2000, seed=0):
    rng=np.random.default_rng(seed); golds=np.asarray(golds_idx); preds=np.asarray(preds)
    N=len(golds); accs=[]; f1s=[]
    for _ in range(n):
        s=rng.integers(0,N,N)
        accs.append(accuracy_score(golds[s],preds[s]))
        f1s.append(f1_score(golds[s],preds[s],labels=list(range(4)),average="macro",zero_division=0))
    q=lambda a:(float(np.percentile(a,2.5)),float(np.percentile(a,97.5)))
    return {"acc_ci":q(accs),"f1_ci":q(f1s)}

def eval_logits(model, ds, shuffle_opts=False):
    rng=random.Random(cfg.seed); prompts=[]; golds=[]
    for i in range(len(ds)):
        ex=ds[i]; opts,ans=ex["options"],ex["answer"]
        if shuffle_opts: opts,ans=shuffle_options(ex["question"],opts,ans,rng)
        prompts.append(format_prompt(ex["question"],opts)); golds.append(OPTIONS.index(ans))
    return student_option_logits(model,prompts).numpy(), golds

def evaluate(model, ds, tag="", shuffle_opts=False, T=1.0):
    logits,golds=eval_logits(model,ds,shuffle_opts)
    m,preds,conf,correct=metrics_from_logits(logits,golds,T)
    if tag:
        print("[%s] acc=%.4f f1=%.4f ece=%.4f%s" % (tag,m["acc"],m["f1"],m["ece"],
              " (shuf)" if shuffle_opts else ""))
        wlog({tag+"/acc":m["acc"],tag+"/f1":m["f1"],tag+"/ece":m["ece"]})
    return m, dict(logits=logits,golds=golds,preds=preds,conf=conf,correct=correct)

def save_reliability(correct, conf, path, title="Reliability", n_bins=10):
    correct=np.asarray(correct,float); conf=np.asarray(conf,float)
    bins=np.linspace(0,1,n_bins+1); xs,ys=[],[]
    for i in range(n_bins):
        lo,hi=bins[i],bins[i+1]
        m=(conf>lo)&(conf<=hi) if i>0 else (conf>=lo)&(conf<=hi)
        if m.sum()==0: continue
        xs.append(conf[m].mean()); ys.append(correct[m].mean())
    plt.figure(figsize=(4,4)); plt.plot([0,1],[0,1],"--",color="gray",label="perfect")
    plt.plot(xs,ys,"o-",label="model"); plt.xlabel("confidence"); plt.ylabel("accuracy")
    plt.title(title); plt.legend(); plt.tight_layout(); plt.savefig(path,dpi=200); plt.close()

def save_confusion(golds_idx, preds_idx, path):
    cm=confusion_matrix(golds_idx,preds_idx,labels=list(range(4)))
    plt.figure(figsize=(4,4)); plt.imshow(cm,cmap="Blues")
    plt.xticks(range(4),OPTIONS); plt.yticks(range(4),OPTIONS)
    plt.xlabel("predicted"); plt.ylabel("gold"); plt.title("Confusion")
    for i in range(4):
        for j in range(4):
            plt.text(j,i,cm[i,j],ha="center",color="white" if cm[i,j]>cm.max()/2 else "black")
    plt.colorbar(); plt.tight_layout(); plt.savefig(path,dpi=200); plt.close()


def save_reliability_combined(corr_raw, conf_raw, corr_cal, conf_cal, ece_raw, ece_cal, path, n_bins=10):
    def curve(correct, conf):
        correct=np.asarray(correct,float); conf=np.asarray(conf,float)
        bins=np.linspace(0,1,n_bins+1); xs,ys=[],[]
        for i in range(n_bins):
            lo,hi=bins[i],bins[i+1]
            m=(conf>lo)&(conf<=hi) if i>0 else (conf>=lo)&(conf<=hi)
            if m.sum()==0: continue
            xs.append(conf[m].mean()); ys.append(correct[m].mean())
        return xs,ys
    xr,yr=curve(corr_raw,conf_raw); xc,yc=curve(corr_cal,conf_cal)
    plt.figure(figsize=(4.2,4.0)); plt.plot([0,1],[0,1],"--",color="gray",label="perfect")
    plt.plot(xr,yr,"o-",label="raw (ECE=%.3f)"%ece_raw)
    plt.plot(xc,yc,"s-",label="T-scaled (ECE=%.3f)"%ece_cal)
    plt.xlabel("confidence"); plt.ylabel("accuracy"); plt.title("Reliability diagram")
    plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(path,dpi=200); plt.close()

def save_conf_hist(conf, correct, path, n_bins=20):
    conf=np.asarray(conf,float); correct=np.asarray(correct,bool)
    plt.figure(figsize=(4.5,3.0))
    plt.hist(conf[correct], bins=n_bins, range=(0,1), alpha=0.6, label="correct")
    plt.hist(conf[~correct], bins=n_bins, range=(0,1), alpha=0.6, label="incorrect")
    plt.xlabel("max-softmax confidence"); plt.ylabel("count"); plt.legend(); plt.title("Confidence histogram")
    plt.tight_layout(); plt.savefig(path,dpi=200); plt.close()


In [ ]:
# === Real GPU power trace (pynvml) ===
class GPUPowerMonitor:
    def __init__(self, interval=2.0):
        self.interval=interval; self.samples=[]; self._stop=False; self._t=None
        try:
            import pynvml; pynvml.nvmlInit(); self.pynvml=pynvml
            self.handles=[pynvml.nvmlDeviceGetHandleByIndex(i)
                          for i in range(pynvml.nvmlDeviceGetCount())]; self.ok=True
        except Exception as e:
            print("pynvml unavailable:",e); self.ok=False
    def _loop(self):
        t0=time.time()
        while not self._stop:
            try: w=[self.pynvml.nvmlDeviceGetPowerUsage(h)/1000.0 for h in self.handles]
            except Exception: w=[]
            self.samples.append(tuple([time.time()-t0]+w)); time.sleep(self.interval)
    def start(self):
        if not self.ok: return
        self._stop=False; self._t=threading.Thread(target=self._loop,daemon=True); self._t.start()
    def stop(self): self._stop=True
    def save(self, path):
        rows=[s for s in self.samples if len(s)>1]
        if not rows: return
        arr=np.array(rows); plt.figure(figsize=(5,3))
        for g in range(1,arr.shape[1]): plt.plot(arr[:,0],arr[:,g],label="GPU"+str(g-1))
        plt.xlabel("time (s)"); plt.ylabel("power (W)"); plt.title("Kaggle T4 power")
        plt.legend(); plt.tight_layout(); plt.savefig(path,dpi=200); plt.close()
        pd.DataFrame(arr,columns=["t"]+["gpu"+str(g) for g in range(arr.shape[1]-1)]
                     ).to_csv(path.replace(".png",".csv"),index=False)
power_mon=GPUPowerMonitor(interval=1.0 if cfg.fast else 3.0); power_mon.start()
print("power monitor:", power_mon.ok)


## 4) Phase 1 - SFT logit classifier

In [ ]:
# === SFT (causal LM, last-token CE, logits_to_keep) ===
from transformers import get_cosine_schedule_with_warmup
from peft import get_peft_model_state_dict, set_peft_model_state_dict

def make_cls_examples(ds, augment=False):
    rng = random.Random(cfg.seed); rows = []
    for i in range(len(ds)):
        ex = ds[i]
        rows.append((format_prompt(ex["question"], ex["options"]), OPTIONS.index(ex["answer"])))
        if augment and cfg.option_shuffle_augment:
            for _ in range(cfg.n_aug_copies):
                o, a = shuffle_options(ex["question"], ex["options"], ex["answer"], rng)
                rows.append((format_prompt(ex["question"], o), OPTIONS.index(a)))
    return rows

class ClsDS(TorchDataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate_cls(b):
    chats = [stu_chat(x[0]) for x in b]; labels = torch.tensor([x[1] for x in b])
    enc = student_tok(chats, return_tensors="pt", padding=True, truncation=True, max_length=cfg.max_src_len)
    return enc, labels

def _trainable(m): return [p for p in m.parameters() if p.requires_grad]

def train_cls(model, rows, epochs, lr, tag):
    if not rows: return
    loader = DataLoader(ClsDS(rows), batch_size=cfg.sft_bs, shuffle=True,
                        collate_fn=collate_cls, num_workers=2, pin_memory=True)
    opt = torch.optim.AdamW(_trainable(model), lr=lr, weight_decay=cfg.sft_weight_decay)
    total = max(1, len(loader)*epochs//cfg.sft_grad_accum)
    sched = get_cosine_schedule_with_warmup(opt, int(total*cfg.sft_warmup_ratio), total)
    model.train(); loss = torch.tensor(0.0)
    for ep in range(epochs):
        if not have_time(): print("  [budget] stop SFT"); break
        for step, (enc, labels) in enumerate(loader):
            enc = {k: v.to(STUDENT_DEVICE) for k, v in enc.items()}; labels = labels.to(STUDENT_DEVICE)
            sel = _letter_logits(_fwd_last(model, enc["input_ids"], enc["attention_mask"])).float()
            loss = F.cross_entropy(sel, labels, label_smoothing=cfg.label_smoothing)
            if not torch.isfinite(loss): opt.zero_grad(set_to_none=True); continue
            (loss/cfg.sft_grad_accum).backward()
            if (step+1) % cfg.sft_grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(_trainable(model), cfg.grad_clip)
                opt.step(); opt.zero_grad(set_to_none=True); sched.step()
        print("  [%s] ep %d/%d loss=%.4f (elapsed %.2fh)" % (tag, ep+1, epochs, loss.item(), elapsed_h()))
    free_gpu()

history = []
# Zero-shot baseline: untrained (LoRA-init) student == base Qwen2.5-3B behaviour
m0v,_ = evaluate(student, val_ds, "phase0_zeroshot_val")
m0t,_ = evaluate(student, test_ds, "phase0_zeroshot_test")
history.append({"phase":"phase0_zeroshot","train":0,"val_acc":m0v["acc"],"test_acc":m0t["acc"],
                "val_f1":m0v["f1"],"test_f1":m0t["f1"],"test_ece":m0t["ece"],"pseudo":0,"pseudo_acc":None})
train_cls(student, make_cls_examples(seed_ds, augment=True), cfg.sft_epochs, cfg.sft_lr, "SFT")
mv,_ = evaluate(student, val_ds, "phase1_val")
mt,_ = evaluate(student, test_ds, "phase1_test")
history.append({"phase":"phase1_sft","train":len(seed_ds),"val_acc":mv["acc"],"test_acc":mt["acc"],
                "val_f1":mv["f1"],"test_f1":mt["f1"],"test_ece":mt["ece"],"pseudo":0,"pseudo_acc":None})
best_val = mv["acc"]
best_state = {k: v.detach().cpu().clone() for k, v in get_peft_model_state_dict(student).items()}


## 5) Teacher (4-bit) - answerer / sampler / verifier / note-writer

In [ ]:
# === Teacher (Qwen-family 4-bit) - reasoning answerer / verifier / note-writer (OOM-safe) ===
from transformers import AutoModelForCausalLM as _AutoCausal, BitsAndBytesConfig as _BnB
t_tok = AutoTokenizer.from_pretrained(cfg.teacher_model, use_fast=True, token=HF_TOKEN)
t_tok.padding_side = "left"; t_tok.truncation_side = "left"
if t_tok.pad_token is None: t_tok.pad_token = t_tok.eos_token
_tq = _BnB(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
           bnb_4bit_compute_dtype=torch.float16)
teacher = _AutoCausal.from_pretrained(cfg.teacher_model, quantization_config=_tq,
          device_map={"": str(TEACHER_DEVICE)}, low_cpu_mem_usage=True, token=HF_TOKEN)
teacher.eval(); print("teacher:", cfg.teacher_model, "| CoT:", cfg.teacher_cot)
T_LETTER_IDS = {}
for L in OPTIONS:
    c = []
    for v in [L, " " + L]:
        e = t_tok.encode(v, add_special_tokens=False)
        if len(e) == 1: c.append(e[0])
    T_LETTER_IDS[L] = c or [t_tok.encode(L, add_special_tokens=False)[0]]

def t_chat(user, system="You are an expert medical doctor taking the USMLE."):
    if hasattr(t_tok, "apply_chat_template"):
        return t_tok.apply_chat_template(
            [{"role":"system","content":system},{"role":"user","content":user}],
            tokenize=False, add_generation_prompt=True)
    return system + "\n\n" + user

def _parse_letter(text):
    """STRICT: accept a letter only if tied to an explicit final-answer marker; else None.
    This rejects truncated / inconclusive chains-of-thought instead of grabbing a distractor."""
    t = text or ""
    pats = [
        r"\\boxed\{\s*([ABCD])\b",
        r"final\s+response[^A-Za-z]{0,20}([ABCD])\b",
        r"final\s+answer[^A-Za-z]{0,20}([ABCD])\b",
        r"correct\s+(?:answer|option|choice)\s*(?:is|:|=)?\s*\(?\s*([ABCD])\b",
        r"\banswer\s*(?:is|:|=)\s*\(?\s*([ABCD])\b",
        r"(?:^|\n)\s*\(?([ABCD])\)?\s*[.)]?\s*$",
    ]
    for p in pats:
        ms = list(re.finditer(p, t, re.I))
        if ms: return ms[-1].group(1).upper()
    return None

def _t_last(input_ids, attention_mask):
    for kw in ("logits_to_keep","num_logits_to_keep"):
        try: return teacher(input_ids=input_ids, attention_mask=attention_mask, **{kw:1}).logits[:, -1, :]
        except TypeError: continue
    return teacher(input_ids=input_ids, attention_mask=attention_mask).logits[:, -1, :]

@torch.no_grad()
def teacher_option_probs(prompts):
    out = []
    for i in range(0, len(prompts), cfg.teacher_score_bs):
        chunk = [t_chat(p) for p in prompts[i:i+cfg.teacher_score_bs]]
        enc = t_tok(chunk, return_tensors="pt", padding=True, truncation=True,
                    max_length=cfg.teacher_max_len).to(TEACHER_DEVICE)
        last = _t_last(enc.input_ids, enc.attention_mask)
        cols = [torch.logsumexp(torch.stack([last[:, t] for t in T_LETTER_IDS[L]], 0), 0) for L in OPTIONS]
        out.append(F.softmax(torch.stack(cols, 1).float(), -1).cpu()); del enc, last
    torch.cuda.empty_cache()
    return torch.cat(out, 0).numpy()

@torch.no_grad()
def teacher_sample_letters(prompt, k):
    mnt = cfg.teacher_cot_tokens if cfg.teacher_cot else cfg.teacher_max_new_tokens
    enc = t_tok([t_chat(prompt)], return_tensors="pt", padding=True, truncation=True,
                max_length=cfg.teacher_max_len).to(TEACHER_DEVICE)
    gen = teacher.generate(**enc, do_sample=True, temperature=cfg.teacher_temperature, top_p=0.9,
                           max_new_tokens=mnt, num_return_sequences=k, pad_token_id=t_tok.pad_token_id)
    texts = t_tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    del enc, gen; torch.cuda.empty_cache()
    return [_parse_letter(tx) for tx in texts]

_VTMPL = ("A junior doctor proposed answer ({cand}). Independently decide the correct option. "
          "{q}\nA. {a}\nB. {b}\nC. {c}\nD. {d}\nReply with ONLY one letter.\nAnswer:")
@torch.no_grad()
def teacher_verify(q, opts, cand):
    p = _VTMPL.format(cand=cand, q=q, a=opts[0], b=opts[1], c=opts[2], d=opts[3])
    return OPTIONS[int(teacher_option_probs([p])[0].argmax())]

@torch.no_grad()
def teacher_note(q, opts, ans):
    p = ("Question: " + q + "\nCorrect answer: " + ans + ". " + opts[OPTIONS.index(ans)] +
         "\nIn <=2 sentences, give the key clinical reasoning (the differentiating feature).")
    enc = t_tok([t_chat(p)], return_tensors="pt", truncation=True, max_length=cfg.teacher_max_len).to(TEACHER_DEVICE)
    gen = teacher.generate(**enc, do_sample=False, max_new_tokens=cfg.teacher_note_tokens,
                           pad_token_id=t_tok.pad_token_id)
    txt = t_tok.decode(gen[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
    del enc, gen; torch.cuda.empty_cache()
    return txt


## 6) Memory, multi-level de-duplication & test-set decontamination
A single embedder powers: the note memory (with near-dup skipping), neighbor retrieval for synthesis, a synthetic-data dedup filter, and a **test decontamination index** that drops any train/synthetic item whose nearest test question exceeds `decon_cos`.

In [ ]:
# === Embeddings, memory, dedup, decontamination (pure-numpy, no chromadb) ===
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(cfg.embed_model, device="cuda:0")
def embed(texts):
    return np.asarray(embedder.encode(list(texts), normalize_embeddings=True,
                                      show_progress_bar=False), dtype=np.float32)

# --- seed neighbor index (for synthesis conditioning) ---
_seed_docs = [seed_ds[i]["question"] + " || " + " | ".join(seed_ds[i]["options"])
              for i in range(len(seed_ds))]
SEED_EMB = embed(_seed_docs)
def retrieve_seed_neighbors(q, k=3):
    if SEED_EMB.shape[0] == 0: return []
    s = SEED_EMB @ embed([q])[0]
    idx = np.argsort(-s)[:min(k, len(_seed_docs))]
    return [_seed_docs[i] for i in idx]

# --- test-set decontamination index ---
TEST_EMB = embed([test_ds[i]["question"] for i in range(len(test_ds))])
def max_test_sim(q): return float((TEST_EMB @ embed([q])[0]).max())
def is_contaminated(q): return cfg.decontaminate and max_test_sim(q) >= cfg.decon_cos

# --- note memory (growing matrix) with active de-duplication ---
_NOTES = {"E": None, "docs": []}
_note_count = {"n": 0, "dups": 0}
def add_note(q, opts, ans, note):
    doc = q + " || " + ans + ": " + note
    e = embed([doc])[0]
    if _NOTES["E"] is not None and float((_NOTES["E"] @ e).max()) >= cfg.dedup_note_cos:
        _note_count["dups"] += 1; return False
    _NOTES["E"] = e[None, :] if _NOTES["E"] is None else np.vstack([_NOTES["E"], e[None, :]])
    _NOTES["docs"].append(doc); _note_count["n"] += 1; return True
def retrieve_notes(q, k=None):
    k = k or cfg.retrieve_k
    if _NOTES["E"] is None: return []
    s = _NOTES["E"] @ embed([q])[0]
    idx = np.argsort(-s)[:min(k, len(_NOTES["docs"]))]
    return [_NOTES["docs"][i] for i in idx]

# --- synthetic-question near-dup filter (seeded with the gold questions) ---
class NearDupFilter:
    def __init__(self, cos): self.cos = cos; self.E = None
    def add_text(self, t):
        e = embed([t])
        self.E = e if self.E is None else np.vstack([self.E, e])
    def is_dup(self, t):
        if self.E is None: return False
        return float((self.E @ embed([t])[0]).max()) >= self.cos
synth_dedup = NearDupFilter(cfg.dedup_synth_cos)
synth_dedup.E = embed([seed_ds[i]["question"] for i in range(len(seed_ds))])
print("memory + dedup ready (numpy) | seed:", SEED_EMB.shape, "| test:", TEST_EMB.shape)


## 7) Difficulty-aware mining (entropy + uncertainty + small margin)

In [ ]:
# === Mining ===
def mine_hard(pool_ds, model, n):
    prompts=[format_prompt(pool_ds[i]["question"],pool_ds[i]["options"]) for i in range(len(pool_ds))]
    probs=student_probs(model,prompts); conf=probs.max(1)
    ent=-(probs*np.log(np.clip(probs,1e-12,1))).sum(1)
    srt=np.sort(probs,1); margin=srt[:,-1]-srt[:,-2]
    score=cfg.mine_alpha*ent+(1-cfg.mine_alpha)*(1-conf)+cfg.mine_margin_w*(1-margin)
    return [int(i) for i in np.argsort(-score)[:n]]


## 8) Three-gate pseudo-labeling + confidence-weighted soft KD

In [ ]:
# === Pseudo-labeling 3-gate (teacher) + causal soft-KD (logits_to_keep) ===
class KDDataset(TorchDataset):
    def __init__(self, p, l, s, w): self.p, self.l, self.s, self.w = p, l, s, w
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.p[i], self.l[i], self.s[i], self.w[i]

def collate_kd(b):
    chats = [stu_chat(x[0]) for x in b]
    labels = torch.tensor([x[1] for x in b])
    soft = torch.tensor(np.stack([x[2] for x in b]), dtype=torch.float32)
    w = torch.tensor([x[3] for x in b], dtype=torch.float32)
    enc = student_tok(chats, return_tensors="pt", padding=True, truncation=True, max_length=cfg.max_src_len)
    return enc, labels, soft, w

def train_kd(model, prompts, labels, soft, weights, epochs, lr, tag):
    if not prompts: return
    loader = DataLoader(KDDataset(prompts, labels, soft, weights), batch_size=cfg.kd_bs, shuffle=True,
                        collate_fn=collate_kd, num_workers=2, pin_memory=True)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=cfg.sft_weight_decay)
    total = max(1, len(loader)*epochs); sched = get_cosine_schedule_with_warmup(opt, int(0.1*total), total)
    T = cfg.kd_temperature; a = cfg.kd_alpha; model.train(); loss = torch.tensor(0.0)
    for ep in range(epochs):
        if not have_time(): print("  [budget] stop KD"); break
        for enc, lb, sf, w in loader:
            enc = {k: v.to(STUDENT_DEVICE) for k, v in enc.items()}
            lb = lb.to(STUDENT_DEVICE); sf = sf.to(STUDENT_DEVICE); w = w.to(STUDENT_DEVICE)
            sel = _letter_logits(_fwd_last(model, enc["input_ids"], enc["attention_mask"])).float()
            ce = F.cross_entropy(sel, lb, label_smoothing=cfg.label_smoothing, reduction="none")
            logp = F.log_softmax(sel/T, -1); tgt = F.softmax(torch.log(torch.clamp(sf, 1e-6))/T, -1)
            kd = F.kl_div(logp, tgt, reduction="none").sum(1) * (T*T)
            loss = ((1-a)*ce + a*w*kd).mean()
            if not torch.isfinite(loss): opt.zero_grad(set_to_none=True); continue
            loss.backward(); torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.grad_clip)
            opt.step(); opt.zero_grad(set_to_none=True); sched.step()
        print("  [%s] ep %d/%d loss=%.4f" % (tag, ep+1, epochs, loss.item()))
    free_gpu()

def run_pseudo_iteration(hard_local_idx, pool_ds, it):
    # NOTE: the batched cell below overrides this; kept as a safe single-prompt fallback.
    acc = {"prompts": [], "labels": [], "soft": [], "w": [], "n_correct": 0}
    n1 = n2 = n3 = ndecon = 0
    for li in hard_local_idx[:cfg.pseudo_max_per_iter]:
        if not have_time(): break
        ex = pool_ds[li]; q, opts = ex["question"], ex["options"]
        if is_contaminated(q): ndecon += 1; continue
        notes = retrieve_notes(q)
        ctx = ("\n\nRelevant notes:\n" + "\n".join("- "+n for n in notes)) if notes else ""
        base = format_prompt(q, opts) + ctx
        samples = teacher_sample_letters(base, cfg.sc_samples)
        valid = [s for s in samples if s in OPTIONS]
        if not valid: continue
        vote, cnt = Counter(valid).most_common(1)[0]
        if cnt < cfg.sc_vote_frac*cfg.sc_samples: continue
        n1 += 1
        if cfg.use_verifier_gate and teacher_verify(q, opts, vote) != vote: continue
        n2 += 1
        soft = teacher_option_probs([base])[0]
        if soft.max() < cfg.conf_gate or OPTIONS[int(soft.argmax())] != vote: continue
        n3 += 1
        acc["prompts"].append(format_prompt(q, opts)); acc["labels"].append(OPTIONS.index(vote))
        acc["soft"].append(soft); acc["w"].append(float(soft.max()) if cfg.kd_conf_weight else 1.0)
        if vote == ex["answer"]: acc["n_correct"] += 1
        add_note(q, opts, vote, teacher_note(q, opts, vote))
    n = len(acc["prompts"]); pacc = (acc["n_correct"]/n) if n else None
    print("  gates SC=%d ->ver=%d ->conf=%d | accepted=%d | decon_skip=%d | PAcc=%s | notes=%d dup=%d"
          % (n1, n2, n3, n, ndecon, ("None" if pacc is None else "%.3f"%pacc), _note_count["n"], _note_count["dups"]))
    return acc, pacc


## 9) Verifier-gated, decontaminated, de-duplicated synthesis

In [ ]:
# === Synthesis ===
SYN_TMPL=("Rewrite the following USMLE question stem so it tests the SAME concept and has the SAME "
          "correct answer, with different wording/patient details. Keep the four options identical. "
          "Context from similar items:\n{ctx}\n\nOriginal: {q}\n\nRewrite (stem only):")
@torch.no_grad()
def synthesize(hard_local_idx, pool_ds, budget):
    out=[]; rejected=0
    for li in hard_local_idx:
        if len(out)>=budget or not have_time(): break
        ex=pool_ds[li]; q,opts,ans=ex["question"],ex["options"],ex["answer"]
        neigh=retrieve_seed_neighbors(q,3); ctx="\n".join("- "+n[:160] for n in neigh) if neigh else "(none)"
        enc=t_tok([t_chat(SYN_TMPL.format(ctx=ctx,q=q))],return_tensors="pt",truncation=True,
                  max_length=1024).to(TEACHER_DEVICE)
        for _ in range(cfg.rewrites_per_q):
            if len(out)>=budget: break
            gen=teacher.generate(**enc,do_sample=True,temperature=0.8,top_p=0.9,max_new_tokens=cfg.synth_max_new_tokens,
                                 pad_token_id=t_tok.pad_token_id)
            nq=t_tok.decode(gen[0,enc.input_ids.shape[1]:],skip_special_tokens=True).strip()
            if len(nq)<20: continue
            if is_contaminated(nq) or synth_dedup.is_dup(nq): rejected+=1; continue
            if teacher_verify(nq,opts,ans)!=ans: rejected+=1; continue
            synth_dedup.add_text(nq); out.append({"question":nq,"options":opts,"answer":ans})
    torch.cuda.empty_cache()
    print("  synthesized=%d (rejected dup/decon/verifier=%d)"%(len(out),rejected))
    return out


## 10) GRPO-KD pass

In [ ]:
# === GRPO-KD (causal; KL anchor via disable_adapter; logits_to_keep) ===
def grpo_kd(model, prompts, targets, tag="grpo"):
    if not cfg.grpo_enable or not prompts or not have_time(): return
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg.grpo_lr)
    G, eps = cfg.grpo_group, 1e-6; idx = list(range(len(prompts))); pg = kl = torch.tensor(0.0)
    for ep in range(cfg.grpo_epochs):
        random.Random(cfg.seed+ep).shuffle(idx)
        for bs in range(0, len(idx), cfg.kd_bs):
            if not have_time(): break
            sub = idx[bs:bs+cfg.kd_bs]
            chats = [stu_chat(prompts[i]) for i in sub]; bt = [targets[i] for i in sub]
            enc = student_tok(chats, return_tensors="pt", padding=True, truncation=True,
                              max_length=cfg.max_src_len).to(STUDENT_DEVICE)
            model.train()
            sel = _letter_logits(_fwd_last(model, enc.input_ids, enc.attention_mask)).float()
            logp = F.log_softmax(sel, -1); probs = logp.exp()
            with torch.no_grad():
                with model.disable_adapter():
                    rsel = _letter_logits(_fwd_last(model, enc.input_ids, enc.attention_mask)).float()
                rlogp = F.log_softmax(rsel, -1)
                samp = torch.multinomial(probs, G, replacement=True)
            tgt = torch.tensor(bt, device=STUDENT_DEVICE).unsqueeze(1)
            reward = (samp == tgt).float()
            adv = (reward - reward.mean(1, keepdim=True)) / (reward.std(1, keepdim=True) + eps)
            pg = -(adv * torch.gather(logp, 1, samp)).mean()
            kl = F.kl_div(logp, rlogp, log_target=True, reduction="batchmean")
            loss = pg + cfg.grpo_kl*kl
            if not torch.isfinite(loss): opt.zero_grad(set_to_none=True); continue
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.grad_clip); opt.step()
        print("  [%s] ep %d pg=%.4f kl=%.4f" % (tag, ep+1, pg.item(), kl.item()))
    free_gpu()


## 11) Iterative loop (budget-guarded)

In [ ]:
# === Pseudo-labeling: FAST forced-answer + option-permutation consistency + confidence ===
import torch, re, numpy as np, random as _random

@torch.no_grad()
def teacher_forced_labels(items, seed=0):
    """For each (q,opts): teacher forced-answer distribution on original AND a permuted order.
    Returns (label_idx, conf, dist[4], perm_agrees) per item."""
    rng = _random.Random(seed)
    p1 = [format_prompt(q, opts) for (q, opts) in items]
    d1 = teacher_option_probs(p1)                       # [n,4], original order
    perms, p2 = [], []
    for (q, opts) in items:
        perm = list(range(4)); rng.shuffle(perm)
        perms.append(perm); p2.append(format_prompt(q, [opts[k] for k in perm]))
    d2 = teacher_option_probs(p2)                       # [n,4], permuted order
    out = []
    for i in range(len(items)):
        a1 = int(d1[i].argmax()); conf = float(d1[i].max())
        a2 = perms[i][int(d2[i].argmax())]             # map permuted argmax back to original index
        out.append((a1, conf, d1[i], a1 == a2))
    return out

@torch.no_grad()
def teacher_note_batch(triples):
    if not triples: return []
    notes = []
    for s in range(0, len(triples), cfg.teacher_score_bs):
        chunk = triples[s:s+cfg.teacher_score_bs]
        ps = [t_chat("Question: "+q+"\nCorrect answer: "+a+". "+o[OPTIONS.index(a)]+
                     "\nIn <=2 sentences, give the key clinical reasoning (the differentiating feature).")
              for (q, o, a) in chunk]
        enc = t_tok(ps, return_tensors="pt", padding=True, truncation=True, max_length=cfg.teacher_max_len).to(TEACHER_DEVICE)
        gen = teacher.generate(**enc, do_sample=False, max_new_tokens=cfg.teacher_note_tokens,
                               pad_token_id=t_tok.pad_token_id)
        notes += [t.strip() for t in t_tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)]
        del enc, gen; torch.cuda.empty_cache()
    return notes

def run_pseudo_iteration(hard_local_idx, pool_ds, it):
    acc = {"prompts": [], "labels": [], "soft": [], "w": [], "n_correct": 0}
    nproc = nperm = ndecon = 0
    items = []
    for li in hard_local_idx[:cfg.pseudo_max_per_iter]:
        ex = pool_ds[li]; q, opts = ex["question"], ex["options"]
        if is_contaminated(q): ndecon += 1; continue
        items.append((ex, q, opts))
    # process in chunks; teacher_option_probs internally batches by teacher_score_bs
    CH = 64
    accepted_meta = []  # (q,opts,letter) for notes
    for s in range(0, len(items), CH):
        if not have_time(): print("  [budget] stop pseudo loop"); break
        chunk = items[s:s+CH]
        res = teacher_forced_labels([(q, opts) for (ex, q, opts) in chunk], seed=cfg.seed+s)
        for (ex, q, opts), (a1, conf, dist, agree) in zip(chunk, res):
            nproc += 1
            if not agree: continue                      # permutation-consistency gate
            nperm += 1
            if conf < cfg.conf_gate: continue           # confidence gate
            letter = OPTIONS[a1]
            acc["prompts"].append(format_prompt(q, opts)); acc["labels"].append(a1)
            acc["soft"].append(dist.astype(np.float32)); acc["w"].append(conf if cfg.kd_conf_weight else 1.0)
            if letter == ex["answer"]: acc["n_correct"] += 1
            accepted_meta.append((q, opts, letter))
        torch.cuda.empty_cache()
    # notes (capped for speed)
    cap = min(len(accepted_meta), cfg.max_notes)
    notes_out = teacher_note_batch([(q, o, a) for (q, o, a) in accepted_meta[:cap]])
    for (q, o, a), note in zip(accepted_meta[:cap], notes_out):
        add_note(q, o, a, note)
    n = len(acc["prompts"]); pacc = (acc["n_correct"]/n) if n else None
    print("  forced+perm gate: processed=%d perm_ok=%d accepted=%d | decon=%d | PAcc=%s | notes=%d"
          % (nproc, nperm, n, ndecon, ("None" if pacc is None else "%.3f"%pacc), _note_count["n"]))
    wlog({"iter%d/accepted"%it: n, "iter%d/pseudo_acc"%it: (pacc or 0.0), "iter%d/notes"%it: _note_count["n"]})
    return acc, pacc
print("Pseudo-labeling: FAST forced-answer + permutation-consistency | conf_gate=%.2f score_bs=%d" %
      (cfg.conf_gate, cfg.teacher_score_bs))


In [ ]:
from peft import get_peft_model_state_dict, set_peft_model_state_dict
# === Orchestration ===
pseudo_bank={"prompts":[],"labels":[],"soft":[],"w":[]}; synth_bank=[]
for it in range(1,cfg.num_iters+1):
    if not have_time(): print("[budget] stopping before iteration",it); break
    print("\n===== ITERATION %d (elapsed %.2fh, left %.2fh) ====="%(it,elapsed_h(),time_left()/3600))
    hard=mine_hard(unlabeled_ds,student,cfg.hard_per_iter)
    ap,pacc=run_pseudo_iteration(hard,unlabeled_ds,it)
    for k in ["prompts","labels","soft","w"]: pseudo_bank[k]+=ap[k]
    synth_bank+=synthesize(hard,unlabeled_ds,cfg.synth_per_iter)
    rows = make_cls_examples(Dataset.from_list(synth_bank), augment=False) if synth_bank else []
    if rows: train_cls(student, rows, 1, cfg.sft_lr, "iter%d_synth_sft"%it)
    if pseudo_bank["prompts"]:
        train_kd(student,pseudo_bank["prompts"],pseudo_bank["labels"],pseudo_bank["soft"],
                 pseudo_bank["w"],cfg.kd_epochs,cfg.kd_lr,"iter%d_kd"%it)
    if cfg.grpo_enable and pseudo_bank["prompts"]:
        grpo_kd(student,pseudo_bank["prompts"][:cfg.grpo_prompts],
                pseudo_bank["labels"][:cfg.grpo_prompts],"iter%d_grpo"%it)
    mv,_=evaluate(student,val_ds,"iter%d_val"%it); mt,_=evaluate(student,test_ds,"iter%d_test"%it)
    history.append({"phase":"iter%d"%it,"train":len(seed_ds)+len(synth_bank),"val_acc":mv["acc"],
                    "test_acc":mt["acc"],"val_f1":mv["f1"],"test_f1":mt["f1"],"test_ece":mt["ece"],
                    "pseudo":len(ap["prompts"]),"pseudo_acc":pacc})
    if mv["acc"]>best_val:
        best_val=mv["acc"]; best_state={k:v.detach().cpu().clone() for k,v in get_peft_model_state_dict(student).items()}
        print("  * best val %.4f"%best_val)
    free_gpu()
_ = set_peft_model_state_dict(student, {k:v.to(STUDENT_DEVICE) for k,v in best_state.items()})
if cfg.ba_enable and have_time():
    print("\n=== Born-again ===")
    sp=[format_prompt(seed_ds[i]["question"],seed_ds[i]["options"]) for i in range(len(seed_ds))]
    soft=student_probs(student,sp); lbl=[OPTIONS.index(seed_ds[i]["answer"]) for i in range(len(seed_ds))]
    train_kd(student,sp,lbl,[soft[i] for i in range(len(sp))],[1.0]*len(sp),cfg.ba_epochs,cfg.ba_lr,"born_again")
    mv,_=evaluate(student,val_ds,"ba_val")
    if mv["acc"]>best_val:
        best_val=mv["acc"]; best_state={k:v.detach().cpu().clone() for k,v in get_peft_model_state_dict(student).items()}
_ = set_peft_model_state_dict(student, {k:v.to(STUDENT_DEVICE) for k,v in best_state.items()})


## 12) Calibration, bootstrap CIs & artifact export

In [ ]:
# === Final eval + calibration + bootstrap + exports + (optional) Hub publish ===
power_mon.stop()
val_logits, val_golds = eval_logits(student, val_ds)
test_logits, test_golds = eval_logits(student, test_ds)
T = 1.0
if cfg.calibrate_temperature:
    T = fit_temperature(val_logits, val_golds); print("fitted temperature T=%.3f" % T)
m_raw, preds_raw, conf_raw, corr_raw = metrics_from_logits(test_logits, test_golds, 1.0)
m_cal, preds_cal, conf_cal, corr_cal = metrics_from_logits(test_logits, test_golds, T)
shuf_logits, shuf_golds = eval_logits(student, test_ds, shuffle_opts=True)
m_shuf,_,_,_ = metrics_from_logits(shuf_logits, shuf_golds, 1.0)
ci = bootstrap_ci(test_golds, preds_raw, n=cfg.bootstrap_n, seed=cfg.seed)
print("\nTEST acc=%.4f f1=%.4f | ECE raw=%.4f -> cal(T=%.2f)=%.4f | Brier=%.4f NLL=%.4f" %
      (m_raw["acc"], m_raw["f1"], m_raw["ece"], T, m_cal["ece"], m_raw["brier"], m_raw["nll"]))
print("acc 95%% CI=[%.4f, %.4f] | f1 95%% CI=[%.4f, %.4f] | shuffled acc=%.4f" %
      (ci["acc_ci"][0], ci["acc_ci"][1], ci["f1_ci"][0], ci["f1_ci"][1], m_shuf["acc"]))

history.append({"phase":"final","train":len(seed_ds)+len(synth_bank),"val_acc":None,
                "test_acc":m_raw["acc"],"val_f1":None,"test_f1":m_raw["f1"],
                "test_ece":m_raw["ece"],"pseudo":None,"pseudo_acc":None})
hist_df = pd.DataFrame(history)
print("\n=== HISTORY ===\n", hist_df.to_string(index=False))
hist_df.to_csv(os.path.join(cfg.out_dir, "history.csv"), index=False)
with open(os.path.join(cfg.out_dir, "history.json"), "w") as f: json.dump(history, f, indent=2)

_,_, conf_v, corr_v = metrics_from_logits(val_logits, val_golds, 1.0)
save_reliability(corr_raw, conf_raw, os.path.join(cfg.out_dir, "reliability_raw.png"), "Reliability (raw)", cfg.ece_bins)
save_reliability(corr_cal, conf_cal, os.path.join(cfg.out_dir, "reliability_calibrated.png"), "Reliability (T-scaled)", cfg.ece_bins)
save_reliability_combined(corr_raw, conf_raw, corr_cal, conf_cal, m_raw["ece"], m_cal["ece"], os.path.join(cfg.out_dir, "reliability_combined.png"), cfg.ece_bins)
save_conf_hist(conf_cal, corr_cal, os.path.join(cfg.out_dir, "confidence_hist.png"))
save_confusion(test_golds, preds_raw, os.path.join(cfg.out_dir, "confusion.png"))
power_mon.save(os.path.join(cfg.out_dir, "gpu_power.png"))
acc_rows = [h for h in history if h["test_acc"] is not None]
plt.figure(figsize=(5,3))
plt.plot([h["phase"] for h in acc_rows], [h["test_acc"] for h in acc_rows], "s-", label="test")
vv = [(h["phase"], h["val_acc"]) for h in acc_rows if h["val_acc"] is not None]
if vv: plt.plot([v[0] for v in vv], [v[1] for v in vv], "o-", label="val")
plt.xticks(rotation=30, ha="right"); plt.ylabel("accuracy"); plt.legend(); plt.title("Accuracy by phase")
plt.tight_layout(); plt.savefig(os.path.join(cfg.out_dir, "accuracy_curve.png"), dpi=200); plt.close()
pa = [(h["phase"], h["pseudo_acc"]) for h in history if h.get("pseudo_acc") is not None]
if pa:
    plt.figure(figsize=(5,3)); plt.plot([p[0] for p in pa], [p[1] for p in pa], "o-")
    plt.axhline(0.25, ls="--", color="gray", label="random"); plt.legend()
    plt.ylabel("oracle PseudoAcc"); plt.title("Pseudo-label accuracy"); plt.xticks(rotation=30, ha="right")
    plt.tight_layout(); plt.savefig(os.path.join(cfg.out_dir, "pseudo_label_acc.png"), dpi=200); plt.close()


# --- selective prediction / abstention (post-hoc on calibrated test confidences) ---
def _risk_coverage(correct, conf):
    correct = np.asarray(correct, float); order = np.argsort(-np.asarray(conf))
    cc = correct[order]; nn = len(cc)
    cov = np.arange(1, nn+1)/nn
    acc_at = np.cumsum(cc)/np.arange(1, nn+1)
    aurc = float((np.trapezoid if hasattr(np,"trapezoid") else np.trapz)(1.0-acc_at, cov))
    return cov, acc_at, aurc
_cov, _acc_at, _aurc = _risk_coverage(corr_cal, conf_cal)
def _acc_at_cov(cv):
    idx = min(len(_cov)-1, max(0, int(cv*len(_cov))-1)); return float(_acc_at[idx])
def _cov_at_acc(tgt):
    idx = np.where(_acc_at >= tgt)[0]
    return float(_cov[idx[-1]]) if len(idx) else 0.0
abstention = {"aurc": _aurc, "acc@50": _acc_at_cov(0.5), "acc@70": _acc_at_cov(0.7),
              "acc@90": _acc_at_cov(0.9), "full_coverage_acc": float(_acc_at[-1]),
              "cov@acc0.80": _cov_at_acc(0.80), "cov@acc0.90": _cov_at_acc(0.90)}
print("Abstention: AURC=%.4f | acc@50%%cov=%.3f acc@70%%cov=%.3f acc@90%%cov=%.3f"
      % (_aurc, abstention["acc@50"], abstention["acc@70"], abstention["acc@90"]))
plt.figure(figsize=(5,3)); plt.plot(_cov, _acc_at, "-")
plt.xlabel("coverage"); plt.ylabel("accuracy on answered"); plt.title("Risk-coverage (selective prediction)")
plt.tight_layout(); plt.savefig(os.path.join(cfg.out_dir, "risk_coverage.png"), dpi=200); plt.close()

with open(os.path.join(cfg.out_dir, "final_metrics.json"), "w") as f:
    json.dump({"test_raw":m_raw,"test_calibrated":m_cal,"temperature":T,"test_shuffled":m_shuf,"abstention":abstention,
               "bootstrap_ci":ci,"best_val_acc":best_val,"notes_stored":_note_count["n"],
               "notes_dup_skipped":_note_count["dups"],"synth_kept":len(synth_bank),
               "elapsed_hours":elapsed_h(),"dataset_used":used_id,"student_model":cfg.student_model,
               "teacher_model":cfg.teacher_model,"history":history}, f, indent=2)

# save + push the LoRA ADAPTER (QLoRA 4-bit base is not merged)
save_dir = os.path.join(cfg.out_dir, "best_student_adapter")
student.save_pretrained(save_dir); student_tok.save_pretrained(save_dir)
def _card():
    return (
"---\nlicense: apache-2.0\nlibrary_name: peft\npipeline_tag: text-generation\n"
"tags:\n- medical\n- medqa\n- usmle\n- knowledge-distillation\n- qlora\n- calibration\n"
"datasets:\n- " + str(used_id).split(' ')[0] + "\nbase_model: " + cfg.student_model + "\n---\n\n"
"# NOTE-RL: Resource-Constrained Medical QA (MedQA-USMLE)\n\n"
"LoRA adapter for **" + cfg.student_model + "**, distilled from **" + cfg.teacher_model +
"** under a **500-label** budget on 2x T4, with three-gate pseudo-labeling, vector note memory, "
"verifier-gated synthesis, GRPO-KD, and temperature-scaled calibration.\n\n"
"## Results (MedQA-USMLE test)\n\n| metric | value |\n|---|---|\n"
"| accuracy | %.4f |\n| macro-F1 | %.4f |\n| ECE (raw) | %.4f |\n"
"| ECE (T=%.2f) | %.4f |\n| acc 95%% CI | [%.4f, %.4f] |\n| option-shuffle acc | %.4f |\n\n"
"## Usage\n```python\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n"
"from peft import PeftModel\nbase = AutoModelForCausalLM.from_pretrained(\"" + cfg.student_model + "\")\n"
"model = PeftModel.from_pretrained(base, \"" + (cfg.hub_model_id or save_dir) + "\")\n"
"tok = AutoTokenizer.from_pretrained(\"" + (cfg.hub_model_id or save_dir) + "\")\n```\n\n"
"_Research artifact; not for clinical use._\n"
) % (m_raw["acc"], m_raw["f1"], m_raw["ece"], T, m_cal["ece"], ci["acc_ci"][0], ci["acc_ci"][1], m_shuf["acc"])
open(os.path.join(save_dir, "README.md"), "w").write(_card())

if cfg.push_to_hub and cfg.hub_model_id and HF_TOKEN:
    try:
        from huggingface_hub import HfApi, upload_file
        HfApi().create_repo(cfg.hub_model_id, token=HF_TOKEN, private=cfg.hub_private, exist_ok=True, repo_type="model")
        student.push_to_hub(cfg.hub_model_id, token=HF_TOKEN, private=cfg.hub_private)
        student_tok.push_to_hub(cfg.hub_model_id, token=HF_TOKEN, private=cfg.hub_private)
        upload_file(path_or_fileobj=os.path.join(save_dir, "README.md"), path_in_repo="README.md",
                    repo_id=cfg.hub_model_id, repo_type="model", token=HF_TOKEN)
        print("Pushed LoRA adapter -> https://huggingface.co/" + cfg.hub_model_id)
    except Exception as e:
        print("push_to_hub failed (need WRITE token + valid hub_model_id):", e)
elif cfg.push_to_hub:
    print("push_to_hub requested but hub_model_id or HF_TOKEN missing; saved locally to", save_dir)

if USE_WANDB and run is not None:
    try: run.finish()
    except Exception: pass
print("\nArtifacts ->", cfg.out_dir, "| elapsed %.2fh" % elapsed_h())


## 15) Label-efficiency sweep
Resets the LoRA adapter and trains **SFT-only** at several seed sizes to produce an accuracy-&-ECE vs label-budget curve. Budget-guarded: it stops and keeps whatever it finished. The headline (4000 + full pipeline) was already saved above; this only adds the curve.

In [ ]:
# === Label-efficiency sweep (additive, budget-guarded) ===
from peft import set_peft_model_state_dict as _spsd, get_peft_model_state_dict as _gpsd2
if cfg.run_efficiency_sweep and have_time(1800):
    sweep_rows = []
    for n in cfg.sweep_sizes:
        if n >= cfg.seed_labeled_n:      # 4000 point already = phase1 headline SFT
            continue
        if not have_time(1800):
            print("  [budget] stop sweep before n=%d" % n); break
        # reset adapter to initialization, then SFT on the first n training items
        _spsd(student, {k: v.to(STUDENT_DEVICE) for k, v in INIT_ADAPTER.items()})
        sub = Dataset.from_list([full_train[i] for i in idx[:n]])
        print("\n--- sweep n=%d (elapsed %.2fh) ---" % (n, elapsed_h()))
        train_cls(student, make_cls_examples(sub, augment=True), cfg.sft_epochs, cfg.sft_lr, "sweep_%d" % n)
        vlog, vg = eval_logits(student, val_ds); tlog, tg = eval_logits(student, test_ds)
        Tn = fit_temperature(vlog, vg) if cfg.calibrate_temperature else 1.0
        mr, pr, cr, kr = metrics_from_logits(tlog, tg, 1.0)
        mc, pc, cc, kc = metrics_from_logits(tlog, tg, Tn)
        cov, acc_at, aurc = _risk_coverage(kc, cc)
        row = {"n_labels": n, "test_acc": mr["acc"], "test_f1": mr["f1"], "ece_raw": mr["ece"],
               "ece_cal": mc["ece"], "brier": mr["brier"], "nll": mr["nll"], "aurc": aurc, "temperature": Tn}
        sweep_rows.append(row)
        print("  sweep n=%d: acc=%.4f f1=%.4f ece_raw=%.4f ece_cal=%.4f aurc=%.4f"
              % (n, mr["acc"], mr["f1"], mr["ece"], mc["ece"], aurc))
        wlog({"sweep/acc": mr["acc"], "sweep/n": n, "sweep/ece_cal": mc["ece"]})
        free_gpu()
    # add the 4000 headline SFT-only point from history (phase1_sft)
    ph1 = next((h for h in history if h["phase"] == "phase1_sft"), None)
    if ph1 is not None:
        sweep_rows.append({"n_labels": cfg.seed_labeled_n, "test_acc": ph1["test_acc"], "test_f1": ph1["test_f1"],
                           "ece_raw": ph1["test_ece"], "ece_cal": None, "brier": None, "nll": None,
                           "aurc": None, "temperature": None})
    sweep_rows = sorted(sweep_rows, key=lambda r: r["n_labels"])
    sdf = pd.DataFrame(sweep_rows)
    print("\n=== EFFICIENCY SWEEP ===\n", sdf.to_string(index=False))
    sdf.to_csv(os.path.join(cfg.out_dir, "efficiency_sweep.csv"), index=False)
    # figure: accuracy & ECE vs labels
    ns = [r["n_labels"] for r in sweep_rows]
    accs = [r["test_acc"] for r in sweep_rows]
    eces = [r["ece_raw"] for r in sweep_rows]
    fig, ax1 = plt.subplots(figsize=(5, 3.2))
    ax1.plot(ns, accs, "o-", color="tab:blue", label="test acc")
    ax1.set_xlabel("labeled seed examples"); ax1.set_ylabel("test accuracy", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax2 = ax1.twinx(); ax2.plot(ns, eces, "s--", color="tab:red", label="ECE (raw)")
    ax2.set_ylabel("ECE", color="tab:red"); ax2.tick_params(axis="y", labelcolor="tab:red")
    plt.title("Label efficiency: accuracy & calibration vs #labels")
    fig.tight_layout(); plt.savefig(os.path.join(cfg.out_dir, "efficiency_curve.png"), dpi=200); plt.close()
    # restore the headline (best) adapter so `student` matches the saved model
    try:
        _spsd(student, {k: v.to(STUDENT_DEVICE) for k, v in best_state.items()})
    except Exception as e:
        print("restore headline adapter skipped:", e)
    print("efficiency sweep done; curve -> efficiency_curve.png")
else:
    print("efficiency sweep disabled or no time budget remaining.")


## 13) (Optional) Ablation harness
Set `cfg.run_ablations=True` to re-evaluate the *trained* student with single components disabled at **inference/eval** time where meaningful (option-shuffle robustness, calibration on/off). Full leave-one-out *training* ablations (GRPO off, verifier off, memory off, mining off) are best run as separate notebook executions by flipping the corresponding `cfg.*` flag and re-running top-to-bottom - each writes its own `final_metrics.json`. This avoids 5x-ing the runtime inside one 12h session.

In [ ]:
# === Ablation notes / quick eval-time variants ===
if cfg.run_ablations:
    rows=[]
    rows.append({"variant":"full (raw)","acc":m_raw["acc"],"f1":m_raw["f1"],"ece":m_raw["ece"]})
    rows.append({"variant":"full (T-scaled)","acc":m_cal["acc"],"f1":m_cal["f1"],"ece":m_cal["ece"]})
    rows.append({"variant":"option-order shuffled","acc":m_shuf["acc"],"f1":m_shuf["f1"],"ece":m_shuf["ece"]})
    abl=pd.DataFrame(rows); print(abl.to_string(index=False))
    abl.to_csv(os.path.join(cfg.out_dir,"ablation_evaltime.csv"),index=False)
    print("\nFor TRAINING ablations re-run the whole notebook after setting ONE of:")
    print("  cfg.grpo_enable=False | cfg.use_verifier_gate=False |")
    print("  cfg.option_shuffle_augment=False | cfg.calibrate_temperature=False |")
    print("  cfg.decontaminate=False | (memory) cfg.retrieve_k=0")
else:
    print("Ablation harness disabled (cfg.run_ablations=False).")


## 14) What to report
- **Headline:** `final_metrics.json` -> `test_raw`, `test_calibrated`, `temperature`, `bootstrap_ci`, `test_shuffled`.
- **Calibration story:** `reliability_raw.png` vs `reliability_calibrated.png` + ECE before/after - directly supports the paper's calibration argument.
- **Iteration table:** `history.csv` (train size, val/test acc & F1, accepted pseudo, oracle PseudoAcc).
- **Figures:** `accuracy_curve.png`, `pseudo_label_acc.png`, `confusion.png`, `gpu_power.png`.
- **Hygiene:** `notes_stored` vs `notes_dup_skipped`, `synth_kept`, and decontamination skips quantify de-dup/leakage control.
- **Ablations:** flip one `cfg.*` flag, re-run, compare `final_metrics.json`.


## 16) UNIQUE CONTRIBUTION - note-conditioned self-consistency + agreement-calibrated abstention
The student reasons (CoT) over retrieved teacher rationales, sampled `reason_samples` times. The **majority vote is the answer** and the **agreement fraction is the confidence**. We compare selective prediction (AURC / accuracy@coverage) using *agreement* confidence vs *softmax* confidence on the same test subset. Additive and budget-guarded - your headline results above are untouched.

In [ ]:
# === UNIQUE: note-conditioned self-consistency reasoning + agreement-calibrated abstention ===
import numpy as np, re
if cfg.run_reasoning_eval and have_time(2400):
    # ---- A) build a rationale memory from GOLD seed (teacher CoT, answers known -> reliable) ----
    Nmem = min(cfg.reason_mem_n, len(seed_ds))
    RMEM_DOCS, RMEM_Q = [], []
    for s0 in range(0, Nmem, cfg.teacher_score_bs):
        if not have_time(2400): break
        chunk = [seed_ds[i] for i in range(s0, min(s0 + cfg.teacher_score_bs, Nmem))]
        ps = [t_chat("Question: " + e["question"] + "\nThe correct answer is (" + e["answer"] + ") " +
                     e["options"][OPTIONS.index(e["answer"])] +
                     ".\nIn 1-2 sentences, give the key clinical reasoning that distinguishes it.")
              for e in chunk]
        enc = t_tok(ps, return_tensors="pt", padding=True, truncation=True, max_length=cfg.teacher_max_len).to(TEACHER_DEVICE)
        gen = teacher.generate(**enc, do_sample=False, max_new_tokens=cfg.teacher_note_tokens, pad_token_id=t_tok.pad_token_id)
        outs = t_tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        for e, r in zip(chunk, outs):
            RMEM_Q.append(e["question"])
            RMEM_DOCS.append("Reasoning: " + r.strip() + " (Answer: " + e["answer"] + ")")
        del enc, gen; torch.cuda.empty_cache()
    RMEM_EMB = embed(RMEM_Q) if RMEM_DOCS else np.zeros((0, 384), dtype=np.float32)
    def retrieve_rationales(q, k):
        if RMEM_EMB.shape[0] == 0: return []
        sims = RMEM_EMB @ embed([q])[0]; idx = np.argsort(-sims)[:k]
        return [RMEM_DOCS[i] for i in idx]
    print("rationale memory:", len(RMEM_DOCS))

    # ---- B) student CoT self-consistency on a test subset ----
    prev_uc = getattr(student.config, "use_cache", False)
    try: student.config.use_cache = True
    except Exception: pass
    try: student.gradient_checkpointing_disable()
    except Exception: pass

    @torch.no_grad()
    def student_cot_vote(q, opts, notes, k):
        ctx = ("Reference notes:\n" + "\n".join("- " + n for n in notes) + "\n\n") if notes else ""
        user = (ctx + "You are a medical expert. Reason step by step, then end with 'Final Answer: X'.\n"
                "Question: " + q + "\nA. " + opts[0] + "\nB. " + opts[1] + "\nC. " + opts[2] + "\nD. " + opts[3])
        chat = stu_chat(user)
        enc = student_tok([chat], return_tensors="pt", truncation=True, max_length=cfg.max_src_len).to(STUDENT_DEVICE)
        gen = student.generate(**enc, do_sample=True, temperature=0.7, top_p=0.9,
                               max_new_tokens=cfg.reason_cot_tokens, num_return_sequences=k,
                               pad_token_id=student_tok.pad_token_id)
        txts = student_tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        letters = [_parse_letter(t) for t in txts]; letters = [x for x in letters if x in OPTIONS]
        if not letters: return None, 0.0
        from collections import Counter as _C
        vote, cnt = _C(letters).most_common(1)[0]
        return vote, cnt / float(k)

    rng = random.Random(cfg.seed)
    tidx = list(range(len(test_ds))); rng.shuffle(tidx); tidx = tidx[:min(cfg.reason_test_n, len(test_ds))]
    sc_pred, sc_conf, gold = [], [], []
    for ii, i in enumerate(tidx):
        if not have_time(900): print("  [budget] stop SC eval at", ii); break
        ex = test_ds[i]; notes = retrieve_rationales(ex["question"], cfg.reason_retrieve_k)
        try:
            v, conf = student_cot_vote(ex["question"], ex["options"], notes, cfg.reason_samples)
        except Exception as e:
            v, conf = None, 0.0
        if v is None: v, conf = "A", 0.0
        sc_pred.append(OPTIONS.index(v)); sc_conf.append(conf); gold.append(OPTIONS.index(ex["answer"]))
    # restore training-mode flags
    try: student.config.use_cache = prev_uc
    except Exception: pass
    try: student.gradient_checkpointing_enable()
    except Exception: pass

    n_eval = len(gold)
    gold = np.array(gold); sc_pred = np.array(sc_pred); sc_conf = np.array(sc_conf)
    sc_acc = float((sc_pred == gold).mean()) if n_eval else 0.0
    sc_correct = (sc_pred == gold).astype(int)
    _, sc_acc_at, sc_aurc = _risk_coverage(sc_correct, sc_conf)

    # ---- C) softmax-confidence baseline on the SAME subset (forced last-token) ----
    sub_prompts = [format_prompt(test_ds[i]["question"], test_ds[i]["options"]) for i in tidx[:n_eval]]
    sm_logits = student_option_logits(student, sub_prompts)
    sm_probs = F.softmax(sm_logits / max(T, 1e-6), -1).numpy()   # temperature-scaled
    sm_pred = sm_probs.argmax(1); sm_conf = sm_probs.max(1)
    sm_correct = (sm_pred == gold).astype(int)
    sm_acc = float((sm_pred == gold).mean())
    _, sm_acc_at, sm_aurc = _risk_coverage(sm_correct, sm_conf)

    def _cov_at(acc_at, cov_arr_len, tgt):
        a = np.asarray(acc_at); idx = np.where(a >= tgt)[0]
        return float((idx[-1] + 1) / len(a)) if len(idx) else 0.0
    res = {
        "n_eval": n_eval, "reason_samples": cfg.reason_samples,
        "self_consistency": {"acc": sc_acc, "aurc": sc_aurc,
                             "cov@acc0.80": _cov_at(sc_acc_at, n_eval, 0.80),
                             "cov@acc0.90": _cov_at(sc_acc_at, n_eval, 0.90)},
        "softmax_baseline": {"acc": sm_acc, "aurc": sm_aurc,
                             "cov@acc0.80": _cov_at(sm_acc_at, n_eval, 0.80),
                             "cov@acc0.90": _cov_at(sm_acc_at, n_eval, 0.90)},
    }
    print("\n=== UNIQUE: self-consistency vs softmax (n=%d) ===" % n_eval)
    print("  SC reasoning : acc=%.4f  AURC=%.4f  (lower AURC = better selective prediction)" % (sc_acc, sc_aurc))
    print("  softmax base : acc=%.4f  AURC=%.4f" % (sm_acc, sm_aurc))
    print("  -> agreement-confidence %s softmax for abstention" %
          ("BEATS" if sc_aurc < sm_aurc else "does NOT beat"))
    with open(os.path.join(cfg.out_dir, "reasoning_eval.json"), "w") as f:
        json.dump(res, f, indent=2)
    # overlay risk-coverage figure
    def _curve(correct, conf):
        order = np.argsort(-np.asarray(conf)); cc = np.asarray(correct)[order]
        return np.arange(1, len(cc)+1)/len(cc), np.cumsum(cc)/np.arange(1, len(cc)+1)
    cxa, cya = _curve(sc_correct, sc_conf); cxb, cyb = _curve(sm_correct, sm_conf)
    plt.figure(figsize=(5, 3.3))
    plt.plot(cxa, cya, "-", label="self-consistency agreement (AURC=%.3f)" % sc_aurc)
    plt.plot(cxb, cyb, "--", label="softmax confidence (AURC=%.3f)" % sm_aurc)
    plt.xlabel("coverage"); plt.ylabel("accuracy on answered"); plt.legend(fontsize=8)
    plt.title("Selective prediction: agreement vs softmax"); plt.tight_layout()
    plt.savefig(os.path.join(cfg.out_dir, "selective_pred_compare.png"), dpi=200); plt.close()
    print("saved reasoning_eval.json + selective_pred_compare.png")
else:
    print("reasoning eval disabled or no time budget.")
